# Module 3: NHS Hospital Waiting Times Performance Analysis

Welcome to **Module 3**! In this lab, you will analyze real NHS England Referral to Treatment (RTT) waiting times data (`NHS_RTT_MAY2026.csv`).

### Public Health Objective:
Identify hospital trusts exceeding 18-week wait targets, uncover specialties with critical backlogs, and produce executive insights.

---

### Assignment Tasks Overview:
1. **Task 1**: Load `NHS_RTT_MAY2026.csv`, inspect schema, and filter out summary rows.
2. **Task 2**: Calculate 18-week compliance percentage across all NHS hospital trusts.
3. **Task 3**: Identify worst-performing medical specialties and critical bottleneck hospitals.
4. **Task 4**: Regional Aggregations & Waiting Times Distribution.
5. **Task 5**: Executive Summary & Actionable Recommendations Report.

In [2]:
# ─── Custom Dataset Loaded: NHS_RTT_MAY2026.csv ───
import pandas as pd

df_custom = pd.read_csv('NHS_RTT_MAY2026.csv')
print(f"Dataset Shape: {df_custom.shape}")
df_custom.head()

Dataset Shape: (178171, 121)



<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Period</th>
      <th>Provider Parent Org Code</th>
      <th>Provider Parent Name</th>
      <th>Provider Org Code</th>
      <th>Provider Org Name</th>
      <th>Commissioner Parent Org Code</th>
      <th>Commissioner Parent Name</th>
      <th>Commissioner Org Code</th>
      <th>Commissioner Org Name</th>
      <th>RTT Part Type</th>
      <th>RTT Part Description</th>
      <th>Treatment Function Code</th>
      <th>Treatment Function Name</th>
      <th>Gt 00 To 01 Weeks SUM 1</th>
      <th>Gt 01 To 02 Weeks SUM 1</th>
      <th>Gt 02 To 03 Weeks SUM 1</th>
      <th>Gt 03 To 04 Weeks SUM 1</th>
      <th>Gt 04 To 05 Weeks SUM 1</th>
      

In [4]:
# Task 1: Inspect schema and remove summary rows

import pandas as pd
import numpy as np

df = pd.read_csv('NHS_RTT_MAY2026.csv', low_memory=False)

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nSummary Rows (C_999):")
print((df['Treatment Function Code'] == 'C_999').sum())

df_clean = df[df['Treatment Function Code'] != 'C_999'].copy()

print("\nShape After Removing Summary Rows:", df_clean.shape)

print("\nRemaining C_999 Rows:",
      (df_clean['Treatment Function Code'] == 'C_999').sum())

print("\nFirst 5 Rows:")
print(df_clean.head())

Dataset Shape: (178171, 121)

Column Names:
['Period', 'Provider Parent Org Code', 'Provider Parent Name', 'Provider Org Code', 'Provider Org Name', 'Commissioner Parent Org Code', 'Commissioner Parent Name', 'Commissioner Org Code', 'Commissioner Org Name', 'RTT Part Type', 'RTT Part Description', 'Treatment Function Code', 'Treatment Function Name', 'Gt 00 To 01 Weeks SUM 1', 'Gt 01 To 02 Weeks SUM 1', 'Gt 02 To 03 Weeks SUM 1', 'Gt 03 To 04 Weeks SUM 1', 'Gt 04 To 05 Weeks SUM 1', 'Gt 05 To 06 Weeks SUM 1', 'Gt 06 To 07 Weeks SUM 1', 'Gt 07 To 08 Weeks SUM 1', 'Gt 08 To 09 Weeks SUM 1', 'Gt 09 To 10 Weeks SUM 1', 'Gt 10 To 11 Weeks SUM 1', 'Gt 11 To 12 Weeks SUM 1', 'Gt 12 To 13 Weeks SUM 1', 'Gt 13 To 14 Weeks SUM 1', 'Gt 14 To 15 Weeks SUM 1', 'Gt 15 To 16 Weeks SUM 1', 'Gt 16 To 17 Weeks SUM 1', 'Gt 17 To 18 Weeks SUM 1', 'Gt 18 To 19 Weeks SUM 1', 'Gt 19 To 20 Weeks SUM 1', 'Gt 20 To 21 Weeks SUM 1', 'Gt 21 To 22 Weeks SUM 1', 'Gt 22 To 23 Weeks SUM 1', 'Gt 23 To 24 Weeks SUM 1'

In [5]:
# Task 2: Calculate 18-week compliance across NHS hospital trusts

# Keep completed pathways only
completed = df_clean[
    df_clean['RTT Part Type'].isin(['Part_1A', 'Part_1B'])
].copy()

# Columns representing waiting times up to 18 weeks
weeks_18 = [
    'Gt 00 To 01 Weeks SUM 1',
    'Gt 01 To 02 Weeks SUM 1',
    'Gt 02 To 03 Weeks SUM 1',
    'Gt 03 To 04 Weeks SUM 1',
    'Gt 04 To 05 Weeks SUM 1',
    'Gt 05 To 06 Weeks SUM 1',
    'Gt 06 To 07 Weeks SUM 1',
    'Gt 07 To 08 Weeks SUM 1',
    'Gt 08 To 09 Weeks SUM 1',
    'Gt 09 To 10 Weeks SUM 1',
    'Gt 10 To 11 Weeks SUM 1',
    'Gt 11 To 12 Weeks SUM 1',
    'Gt 12 To 13 Weeks SUM 1',
    'Gt 13 To 14 Weeks SUM 1',
    'Gt 14 To 15 Weeks SUM 1',
    'Gt 15 To 16 Weeks SUM 1',
    'Gt 16 To 17 Weeks SUM 1',
    'Gt 17 To 18 Weeks SUM 1'
]

# Convert waiting-time columns to numeric
completed[weeks_18] = completed[weeks_18].apply(
    pd.to_numeric, errors='coerce'
)

completed['Total'] = pd.to_numeric(
    completed['Total'], errors='coerce'
)

# Patients waiting up to 18 weeks
completed['Within_18_Weeks'] = completed[weeks_18].sum(axis=1)

# Aggregate by NHS hospital trust
trust_compliance = (
    completed
    .groupby(
        ['Provider Org Code', 'Provider Org Name'],
        as_index=False
    )
    .agg(
        Within_18_Weeks=('Within_18_Weeks', 'sum'),
        Total_Completed=('Total', 'sum')
    )
)

# Calculate compliance percentage
trust_compliance['Compliance_%'] = (
    trust_compliance['Within_18_Weeks']
    / trust_compliance['Total_Completed']
    * 100
)

# Sort from highest to lowest compliance
trust_compliance = trust_compliance.sort_values(
    'Compliance_%',
    ascending=False
)

print("Number of NHS Hospital Trusts:",
      len(trust_compliance))

print("\n18-Week Compliance by Trust:")
print(trust_compliance.head(20).to_string(index=False))

Number of NHS Hospital Trusts: 517

18-Week Compliance by Trust:
Provider Org Code                                                     Provider Org Name  Within_18_Weeks  Total_Completed  Compliance_%
              AJ8                                          THE GRANGE MEDICAL CENTRE HQ            144.0            144.0         100.0
            V8A3H           SPAMEDICA SHEFFIELD COMMUNITY DIAGNOSTIC & TREATMENT CENTRE              5.0              5.0         100.0
            K3V9U           SPAMEDICA STEVENAGE COMMUNITY DIAGNOSTIC & TREATMENT CENTRE              2.0              2.0         100.0
              ACM                                                  EYE CARE MEDICAL LTD             27.0             27.0         100.0
            J0M7P                    SPAMEDICA HEREFORD DIAGNOSTIC AND TREATMENT CENTRE              5.0              5.0         100.0
            M6L9D                                     NEWMEDICA - NOTTINGHAM - KEYWORTH             10.0             10

In [6]:
# Task 3: Identify worst-performing specialties and bottleneck hospitals

# -------------------------------
# A. Specialty-level performance
# -------------------------------

specialty_summary = (
    completed
    .groupby(
        ['Treatment Function Code', 'Treatment Function Name'],
        as_index=False
    )
    .agg(
        Within_18_Weeks=('Within_18_Weeks', 'sum'),
        Total_Completed=('Total', 'sum')
    )
)

specialty_summary['Compliance_%'] = (
    specialty_summary['Within_18_Weeks']
    / specialty_summary['Total_Completed']
    * 100
)

# Remove specialties with no completed pathways
specialty_summary = specialty_summary[
    specialty_summary['Total_Completed'] > 0
].copy()

# Lowest compliance first
worst_specialties = specialty_summary.sort_values(
    'Compliance_%',
    ascending=True
)

print("=== Worst-Performing Specialties ===")
print(
    worst_specialties.head(15).to_string(index=False)
)


# -------------------------------
# B. Hospital bottlenecks
# -------------------------------

# Add patients waiting MORE than 18 weeks
completed['Over_18_Weeks'] = (
    completed['Total'] -
    completed['Within_18_Weeks']
)

hospital_bottlenecks = (
    completed
    .groupby(
        ['Provider Org Code', 'Provider Org Name'],
        as_index=False
    )
    .agg(
        Within_18_Weeks=('Within_18_Weeks', 'sum'),
        Total_Completed=('Total', 'sum'),
        Over_18_Weeks=('Over_18_Weeks', 'sum')
    )
)

hospital_bottlenecks['Compliance_%'] = (
    hospital_bottlenecks['Within_18_Weeks']
    / hospital_bottlenecks['Total_Completed']
    * 100
)

# Percentage of patients waiting over 18 weeks
hospital_bottlenecks['Over_18_%'] = (
    hospital_bottlenecks['Over_18_Weeks']
    / hospital_bottlenecks['Total_Completed']
    * 100
)

# Focus on hospitals with meaningful patient volume
hospital_bottlenecks = hospital_bottlenecks[
    hospital_bottlenecks['Total_Completed'] >= 100
].copy()

# Highest number of patients waiting >18 weeks
hospital_bottlenecks = hospital_bottlenecks.sort_values(
    'Over_18_Weeks',
    ascending=False
)

print("\n=== Critical Bottleneck Hospitals ===")
print(
    hospital_bottlenecks.head(15).to_string(index=False)
)

=== Worst-Performing Specialties ===
Treatment Function Code        Treatment Function Name  Within_18_Weeks  Total_Completed  Compliance_%
                  C_140           Oral Surgery Service          24217.0          48434.0     50.000000
                  C_400              Neurology Service          14839.0          29529.0     50.252294
                  C_110 Trauma and Orthopaedic Service          75285.0         140429.0     53.610721
                  C_120    Ear Nose and Throat Service          49582.0          91854.0     53.979141
                  C_150          Neurosurgical Service           5554.0           9271.0     59.907238
                  C_502            Gynaecology Service          58061.0          94285.0     61.580315
                  C_320             Cardiology Service          39376.0          62781.0     62.719613
                    X04    Other - Paediatric Services          40951.0          63073.0     64.926355
                  C_100        Gener

In [7]:
# Task 4: Regional Aggregation & Waiting Time Distribution

# --------------------------------
# 1. Regional aggregation
# --------------------------------

regional_summary = (
    completed
    .groupby(
        ['Provider Parent Org Code', 'Provider Parent Name'],
        as_index=False
    )
    .agg(
        Within_18_Weeks=('Within_18_Weeks', 'sum'),
        Total_Completed=('Total', 'sum'),
        Over_18_Weeks=('Over_18_Weeks', 'sum')
    )
)

regional_summary['Compliance_%'] = (
    regional_summary['Within_18_Weeks']
    / regional_summary['Total_Completed']
    * 100
)

regional_summary['Over_18_%'] = (
    regional_summary['Over_18_Weeks']
    / regional_summary['Total_Completed']
    * 100
)

regional_summary = regional_summary.sort_values(
    'Compliance_%',
    ascending=True
)

print("=== Regional Aggregation ===")
print(
    regional_summary.to_string(index=False)
)


# --------------------------------
# 2. Waiting-time distribution
# --------------------------------

waiting_columns = [
    'Gt 00 To 01 Weeks SUM 1',
    'Gt 01 To 02 Weeks SUM 1',
    'Gt 02 To 03 Weeks SUM 1',
    'Gt 03 To 04 Weeks SUM 1',
    'Gt 04 To 05 Weeks SUM 1',
    'Gt 05 To 06 Weeks SUM 1',
    'Gt 06 To 07 Weeks SUM 1',
    'Gt 07 To 08 Weeks SUM 1',
    'Gt 08 To 09 Weeks SUM 1',
    'Gt 09 To 10 Weeks SUM 1',
    'Gt 10 To 11 Weeks SUM 1',
    'Gt 11 To 12 Weeks SUM 1',
    'Gt 12 To 13 Weeks SUM 1',
    'Gt 13 To 14 Weeks SUM 1',
    'Gt 14 To 15 Weeks SUM 1',
    'Gt 15 To 16 Weeks SUM 1',
    'Gt 16 To 17 Weeks SUM 1',
    'Gt 17 To 18 Weeks SUM 1',
    'Gt 18 To 19 Weeks SUM 1',
    'Gt 19 To 20 Weeks SUM 1',
    'Gt 20 To 21 Weeks SUM 1',
    'Gt 21 To 22 Weeks SUM 1',
    'Gt 22 To 23 Weeks SUM 1',
    'Gt 23 To 24 Weeks SUM 1',
    'Gt 24 To 25 Weeks SUM 1',
    'Gt 25 To 26 Weeks SUM 1',
    'Gt 26 To 27 Weeks SUM 1'
]

waiting_distribution = (
    completed[waiting_columns]
    .sum()
    .reset_index()
)

waiting_distribution.columns = [
    'Waiting_Week_Bucket',
    'Patients'
]

print("\n=== Waiting Time Distribution ===")
print(waiting_distribution.to_string(index=False))

=== Regional Aggregation ===
Provider Parent Org Code                                                          Provider Parent Name  Within_18_Weeks  Total_Completed  Over_18_Weeks  Compliance_%  Over_18_%
                     QK1               NHS LEICESTER, LEICESTERSHIRE AND RUTLAND INTEGRATED CARE BOARD          12640.0          22694.0        10054.0     55.697541  44.302459
                     QJM                                        NHS LINCOLNSHIRE INTEGRATED CARE BOARD           5663.0          10010.0         4347.0     56.573427  43.426573
                   D7T5G                                               NHS ESSEX INTEGRATED CARE BOARD          30210.0          51397.0        21187.0     58.777750  41.222250
                     QPM                                    NHS NORTHAMPTONSHIRE INTEGRATED CARE BOARD           8621.0          14203.0         5582.0     60.698444  39.301556
                     QT1                      NHS NOTTINGHAM AND NOTTINGHAMSHIRE INTEG

In [8]:
# Task 5: Executive Summary & Actionable Recommendations

# Overall performance
total_within_18 = completed['Within_18_Weeks'].sum()
total_completed = completed['Total'].sum()
total_over_18 = completed['Over_18_Weeks'].sum()

overall_compliance = (
    total_within_18 / total_completed * 100
)

overall_over_18 = (
    total_over_18 / total_completed * 100
)

# Worst specialty
worst_specialty = worst_specialties.iloc[0]

# Largest hospital bottleneck
largest_bottleneck = hospital_bottlenecks.iloc[0]

# Lowest regional compliance
lowest_region = regional_summary.iloc[0]

# Highest regional compliance
highest_region = regional_summary.iloc[-1]


print("========== NHS RTT EXECUTIVE SUMMARY ==========")

print(f"\nTotal Completed Pathways: {total_completed:,.0f}")
print(f"Within 18 Weeks: {total_within_18:,.0f}")
print(f"Over 18 Weeks: {total_over_18:,.0f}")
print(f"Overall 18-Week Compliance: {overall_compliance:.2f}%")
print(f"Patients Over 18 Weeks: {overall_over_18:.2f}%")

print("\n--- Worst-Performing Specialty ---")
print(
    f"{worst_specialty['Treatment Function Name']} "
    f"({worst_specialty['Compliance_%']:.2f}% compliance)"
)

print("\n--- Largest Hospital Bottleneck ---")
print(
    f"{largest_bottleneck['Provider Org Name']} "
    f"({largest_bottleneck['Over_18_Weeks']:,.0f} patients over 18 weeks)"
)

print("\n--- Lowest Regional Compliance ---")
print(
    f"{lowest_region['Provider Parent Name']} "
    f"({lowest_region['Compliance_%']:.2f}%)"
)

print("\n--- Highest Regional Compliance ---")
print(
    f"{highest_region['Provider Parent Name']} "
    f"({highest_region['Compliance_%']:.2f}%)"
)

print("\n========== ACTIONABLE RECOMMENDATIONS ==========")

print("""
1. Prioritise specialties with the lowest 18-week compliance,
   particularly Oral Surgery, Neurology, Trauma & Orthopaedics,
   and ENT.

2. Focus additional capacity and waiting-list recovery efforts
   on hospitals with the highest number of patients waiting over
   18 weeks.

3. Review regional variation and investigate operational practices
   in lower-compliance Integrated Care Board areas.

4. Monitor long-wait patient cohorts separately, especially
   patients waiting beyond 18 weeks.

5. Use regular trust-level and specialty-level monitoring to
   identify deterioration early and track improvement.
""")

========== NHS RTT EXECUTIVE SUMMARY ==========

Total Completed Pathways: 1,405,452
Within 18 Weeks: 938,340
Over 18 Weeks: 467,112
Overall 18-Week Compliance: 66.76%
Patients Over 18 Weeks: 33.24%

--- Worst-Performing Specialty ---
Oral Surgery Service (50.00% compliance)

--- Largest Hospital Bottleneck ---
MANCHESTER UNIVERSITY NHS FOUNDATION TRUST (11,961 patients over 18 weeks)

--- Lowest Regional Compliance ---
NHS LEICESTER, LEICESTERSHIRE AND RUTLAND INTEGRATED CARE BOARD (55.70%)

--- Highest Regional Compliance ---
NHS CORNWALL AND THE ISLES OF SCILLY INTEGRATED CARE BOARD (77.40%)

========== ACTIONABLE RECOMMENDATIONS ==========

1. Prioritise specialties with the lowest 18-week compliance,
   particularly Oral Surgery, Neurology, Trauma & Orthopaedics,
   and ENT.

2. Focus additional capacity and waiting-list recovery efforts
   on hospitals with the highest number of patients waiting over
   18 weeks.

3. Review regional variation and investigate operational practices